# Feather verdicts: hand checks for the per-feather checks

Your verdicts on a fixed sample of feathers, to measure how often each per-feather check is right, and whether a
change to the checks makes them better or worse. One feather at a time:

- **Left:** the panel the checks look at (the slide around the feather with the mask outline, and the pixels inside
  it). **Right:** the saved crop, the image you actually get. Click either to open it full size.
- Judge the **saved crop**: is it usable for measuring this feather?
  - **Good** saves and moves on.
  - **Bad:** tick what is wrong (as many as apply), optionally add a note, then **Save & next**.
- Verdicts are saved after every click to `data/review/verdicts_<run>.csv`. Close the notebook any time and come
  back later: it starts at the first feather without a verdict.

The check results are hidden on purpose, so they don't steer your eye. Kernel: **bird-feather-analysis (pixi)**.

## Parameters

In [ ]:
RUN = "slice100-frag-20261019"     # the run whose feathers are judged
N_FAILED, N_UNCERTAIN, N_PASSED = 13, 10, 18   # random sample sizes per group (fixed seed)
SEED = 7
# verdicts you already gave in messages: crop -> (verdict, problems)
KNOWN = {
    "A3944_2006_Feather_B.jpg": ("good", []),
    "BD31002_1995_Feather_A.jpg": ("good", []),
    "BD31002_1995_Feather_B.jpg": ("good", []),
    "BX40066_20080520_Feather_A.jpg": ("good", []),
    "A3818_2002_Feather_C.jpg": ("bad", ["cut: left"]),
    "AX87168_1990_Feather_B.jpg": ("bad", ["glue or background inside"]),
    "BB17244_19920621_Feather_E.jpg": ("bad", ["partly hidden"]),
    "BB17692_19980621-im0515_Feather_A.jpg": ("bad", ["mostly hidden"]),
    "BB17692_19980621-im0515_Feather_B.jpg": ("bad", ["mostly hidden", "glue or background inside"]),
}

## The sample and the verdict file

In [ ]:
import base64, csv, io, sys
from pathlib import Path
from urllib.parse import quote
sys.path.insert(0, "..")
import pandas as pd
import ipywidgets as w
from IPython.display import display
from PIL import Image

PROBLEMS = ["cut: left", "cut: right", "cut: top", "cut: bottom", "holes or gaps", "glue or background inside",
            "other feather inside", "partly hidden", "mostly hidden", "not a whole feather (piece)", "other"]
SEG = Path("../data/sample_runs") / RUN
REVIEW = Path("../data/review"); REVIEW.mkdir(parents=True, exist_ok=True)
SAMPLE = REVIEW / f"verdict_sample_{RUN}.csv"
VERDICTS = REVIEW / f"verdicts_{RUN}.csv"
COLS = ["crop", "file_id", "feather_id", "verdict", *PROBLEMS, "note", "source"]

if not SAMPLE.exists():          # fixed once, so verdicts stay comparable across changes
    fdf = pd.read_csv(Path("../data/label_runs") / RUN / "feathers.csv")
    seg = fdf[fdf.segment_status == "segmented"].fillna({"needs_review": "", "review_uncertain": ""})
    seg = seg[~seg.crop.isin(KNOWN)]
    pick = pd.concat([
        seg[seg.needs_review != ""].sample(N_FAILED, random_state=SEED),
        seg[(seg.needs_review == "") & (seg.review_uncertain != "")].sample(N_UNCERTAIN, random_state=SEED),
        seg[(seg.needs_review == "") & (seg.review_uncertain == "")].sample(N_PASSED, random_state=SEED)])
    known = fdf[fdf.crop.isin(KNOWN)]
    sample = pd.concat([known, pick]).sample(frac=1, random_state=SEED)   # shuffled: groups don't show
    sample[["crop", "file_id", "feather_id", "qc_image"]].to_csv(SAMPLE, index=False)
sample = pd.read_csv(SAMPLE).fillna("")

verdicts = {}
if VERDICTS.exists():
    verdicts = {r["crop"]: r for r in csv.DictReader(open(VERDICTS, newline=""))}
for crop, (v, probs) in KNOWN.items():
    if crop not in verdicts and crop in set(sample.crop):
        row = sample[sample.crop == crop].iloc[0]
        verdicts[crop] = {"crop": crop, "file_id": row.file_id, "feather_id": row.feather_id, "verdict": v,
                          **{p: ("x" if p in probs else "") for p in PROBLEMS}, "note": "", "source": "your message"}

def save():
    with open(VERDICTS, "w", newline="") as fh:
        wr = csv.DictWriter(fh, fieldnames=COLS)
        wr.writeheader()
        for c in sample.crop:
            if c in verdicts:
                wr.writerow({k: verdicts[c].get(k, "") for k in COLS})
save()
print(f"{len(sample)} feathers, {len(verdicts)} with a verdict -> {VERDICTS}")

## Judge

In [ ]:
def _png(path, h):
    im = Image.open(path).convert("RGB"); im.thumbnail((10_000, h))
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=85); return buf.getvalue()

def _link(path, label):
    rel = quote(str(Path("..") / Path(path).resolve().relative_to(Path("..").resolve())))
    return f'<a href="{rel}" target="_blank">{label}</a>'

state = {"i": next((k for k, c in enumerate(sample.crop) if c not in verdicts), 0)}
title, imgs, links = w.HTML(), w.HBox(), w.HTML()
toggles = [w.ToggleButton(description=p, layout=w.Layout(width="auto")) for p in PROBLEMS]
note = w.Text(placeholder="note (optional)", layout=w.Layout(width="60%"))
good, bad = w.Button(description="Good", button_style="success"), w.Button(description="Save & next (bad)", button_style="danger")
prev, nxt = w.Button(description="< Prev"), w.Button(description="Next >")
progress = w.HTML()

def show():
    i = state["i"]; r = sample.iloc[i]
    v = verdicts.get(r.crop, {})
    title.value = (f"<h3 style='margin:4px 0'>{i + 1} / {len(sample)}: {r.feather_id or r.crop}</h3>"
                   + (f"<span style='color:#2e8b57'>saved: <b>{v['verdict']}</b>"
                      + (f" ({', '.join(p for p in PROBLEMS if v.get(p))})" if v.get("verdict") == "bad" else "")
                      + (f" - {v['source']}" if v.get("source") else "") + "</span>" if v else ""))
    qc, crop = SEG / r.qc_image, SEG / r.crop
    imgs.children = [w.Image(value=_png(qc, 520), format="jpeg"), w.Image(value=_png(crop, 520), format="jpeg")]
    links.value = f"open full size: {_link(qc, 'check panel')} · {_link(crop, 'saved crop')}"
    for t in toggles:
        t.value = bool(v.get(t.description))
    note.value = v.get("note", "")
    done = sum(c in verdicts for c in sample.crop)
    progress.value = f"<b>{done} / {len(sample)}</b> done" + (" - all done, thank you!" if done == len(sample) else "")

def record(verdict):
    r = sample.iloc[state["i"]]
    probs = [t.description for t in toggles if t.value] if verdict == "bad" else []
    verdicts[r.crop] = {"crop": r.crop, "file_id": r.file_id, "feather_id": r.feather_id, "verdict": verdict,
                        **{p: ("x" if p in probs else "") for p in PROBLEMS}, "note": note.value, "source": "notebook"}
    save()
    state["i"] = min(state["i"] + 1, len(sample) - 1); show()

def on_bad(_):
    if not any(t.value for t in toggles) and not note.value.strip():
        progress.value = "<span style='color:#c0392b'>tick what is wrong (or add a note) first</span>"; return
    record("bad")

good.on_click(lambda _: record("good")); bad.on_click(on_bad)
prev.on_click(lambda _: (state.update(i=max(0, state["i"] - 1)), show()))
nxt.on_click(lambda _: (state.update(i=min(len(sample) - 1, state["i"] + 1)), show()))
show()
display(w.VBox([title, imgs, links, w.HBox([good]), w.HTML("<b>Bad:</b> what is wrong with the saved crop?"),
                w.HBox(toggles, layout=w.Layout(flex_flow="row wrap")), w.HBox([note, bad]),
                w.HBox([prev, nxt, progress])]))

## Afterwards
The verdicts file is all that's needed; nothing else to run here. Claude compares each check with your
verdicts (missed problems vs false alarms) before and after every change to the checks.